# Modelling to Generate Alternatives (MGA) for a multi-regional energy system

The cost-optimal design of an energy system is rarely the only sensible one. Often, many designs whose costs are only slightly higher differ considerably in their technology mix. Modelling to Generate Alternatives (MGA) explores this near-optimal solution space. In ETHOS.FINE, MGA is available with the function `fn.runMGA`, which works in four steps:
1. The cost-optimal solution is computed with `esM.optimize`.
2. A constraint limits the total annual cost to the optimal cost plus a slack (e.g. 10 %).
3. The cost objective is replaced by a weighted sum of capacity and/or operation variables, which is either minimized or maximized.
4. The already declared optimization problem is solved again for every alternative.

The multi-regional energy system of example 03 is used here. The workflow is structured as follows:
1. Required packages are imported and the input data path is set
2. An energy system model instance is created
3. Commodity sources are added to the energy system model
4. Commodity conversion components are added to the energy system model
5. Commodity storages are added to the energy system model
6. Commodity transmission components are added to the energy system model
7. Commodity sinks are added to the energy system model
8. The time series are aggregated
9. The cost-optimal solution is computed
10. Alternatives are generated with MGA
11. The alternatives are compared

# 1. Import required packages and set input data path

The ETHOS.FINE framework is imported which provides the required classes and functions for modeling the energy system.

In [1]:
import fine as fn
from getData import getData
import pandas as pd

data = getData()

%matplotlib inline
%load_ext autoreload
%autoreload 2

# 2. Create an energy system model instance 

The structure of the energy system model is given by the considered locations, commodities, the number of time steps as well as the hours per time step.

The commodities are specified by a unit (i.e. 'GW_electric', 'GW_H2lowerHeatingValue', 'Mio. t CO2/h') which can be given as an energy or mass unit per hour. Furthermore, the cost unit and length unit are specified.

In [2]:
locations = {
    "cluster_0",
    "cluster_1",
    "cluster_2",
    "cluster_3",
    "cluster_4",
    "cluster_5",
    "cluster_6",
    "cluster_7",
}
commodityUnitDict = {
    "electricity": r"GW$_{el}$",
    "methane": r"GW$_{CH_{4},LHV}$",
    "biogas": r"GW$_{biogas,LHV}$",
    "CO2": r"Mio. t$_{CO_2}$/h",
    "hydrogen": r"GW$_{H_{2},LHV}$",
}
commodities = {"electricity", "hydrogen", "methane", "biogas", "CO2"}
numberOfTimeSteps = 8760
hoursPerTimeStep = 1
CO2_reductionTarget = 1
balanceLimit = pd.DataFrame(
    index=["CO2 limit"],
    columns=["Total", "lowerBound"],
    data=[[-366 * (1 - CO2_reductionTarget), True]],
)

In [3]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    numberOfTimeSteps=8760,
    commodityUnitsDict=commodityUnitDict,
    hoursPerTimeStep=1,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
    balanceLimit=balanceLimit,
)

# 3. Add commodity sources to the energy system model

## 3.1. Electricity sources

### Wind onshore

In [4]:
esM.add(
    fn.Source(
        esM=esM,
        name="Wind (onshore)",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (onshore), operationRateMax"],
        capacityMax=data["Wind (onshore), capacityMax"],
        investPerCapacity=1.1,
        opexPerCapacity=1.1 * 0.02,
        interestRate=0.08,
        economicLifetime=20,
    )
)

Full load hours:

In [5]:
data["Wind (onshore), operationRateMax"].sum()

cluster_0    1572.003960
cluster_1    2350.292663
cluster_2    2374.507270
cluster_3    2186.572278
cluster_4    1572.650655
cluster_5    1767.840650
cluster_6    2719.564564
cluster_7    1553.045964
dtype: float64

### Wind offshore

In [6]:
esM.add(
    fn.Source(
        esM=esM,
        name="Wind (offshore)",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (offshore), operationRateMax"],
        capacityMax=data["Wind (offshore), capacityMax"],
        investPerCapacity=2.3,
        opexPerCapacity=2.3 * 0.02,
        interestRate=0.08,
        economicLifetime=20,
    )
)

Full load hours:

In [7]:
data["Wind (offshore), operationRateMax"].sum()

cluster_0       0.000000
cluster_1    4435.420314
cluster_2    4301.655834
cluster_3    3902.391858
cluster_4       0.000000
cluster_5       0.000000
cluster_6    4609.508396
cluster_7       0.000000
dtype: float64

### PV

In [8]:
esM.add(
    fn.Source(
        esM=esM,
        name="PV",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["PV, operationRateMax"],
        capacityMax=data["PV, capacityMax"],
        investPerCapacity=0.65,
        opexPerCapacity=0.65 * 0.02,
        interestRate=0.08,
        economicLifetime=25,
    )
)

Full load hours:

In [9]:
data["PV, operationRateMax"].sum()

cluster_0    1113.216464
cluster_1    1053.579422
cluster_2    1058.005181
cluster_3    1079.872237
cluster_4    1140.407380
cluster_5    1051.848141
cluster_6    1069.843344
cluster_7    1085.697466
dtype: float64

### Exisisting run-of-river hydroelectricity plants

In [10]:
esM.add(
    fn.Source(
        esM=esM,
        name="Existing run-of-river plants",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateFix=data["Existing run-of-river plants, operationRateFix"],
        tsaWeight=0.01,
        capacityFix=data["Existing run-of-river plants, capacityFix"],
        investPerCapacity=0,
        opexPerCapacity=0.208,
    )
)

## 3.2. Methane (natural gas and biogas)

### Natural gas

In [11]:
esM.add(
    fn.Source(
        esM=esM,
        name="Natural gas purchase",
        commodity="methane",
        hasCapacityVariable=False,
        commodityCost=0.0331 * 1e-3,
    )
)

### Biogas

In [12]:
esM.add(
    fn.Source(
        esM=esM,
        name="Biogas purchase",
        commodity="biogas",
        operationRateMax=data["Biogas, operationRateMax"],
        hasCapacityVariable=False,
        commodityCost=0.05409 * 1e-3,
    )
)

# 4. Add conversion components to the energy system model

### Combined cycle gas turbine plants

In [13]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="CCGT plants (methane)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={
            "electricity": 1,
            "methane": -1 / 0.6,
            "CO2": 201 * 1e-6 / 0.6,
        },
        hasCapacityVariable=True,
        investPerCapacity=0.65,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

### New combined cycle gas turbine plants for biogas

In [14]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="New CCGT plants (biogas)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": 1, "biogas": -1 / 0.63},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

### New combined cycly gas turbines for hydrogen

In [15]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="New CCGT plants (hydrogen)",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": 1, "hydrogen": -1 / 0.63},
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

### Electrolyzers

In [16]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Electrolyzer",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.7},
        hasCapacityVariable=True,
        investPerCapacity=0.5,
        opexPerCapacity=0.5 * 0.025,
        interestRate=0.08,
        economicLifetime=10,
    )
)

### rSOC

In [17]:
capexRSOC = 1.5

esM.add(
    fn.Conversion(
        esM=esM,
        name="rSOEC",
        physicalUnit=r"GW$_{el}$",
        linkedConversionCapacityID="rSOC",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.6},
        hasCapacityVariable=True,
        investPerCapacity=capexRSOC / 2,
        opexPerCapacity=capexRSOC * 0.02 / 2,
        interestRate=0.08,
        economicLifetime=10,
    )
)

esM.add(
    fn.Conversion(
        esM=esM,
        name="rSOFC",
        physicalUnit=r"GW$_{el}$",
        linkedConversionCapacityID="rSOC",
        commodityConversionFactors={"electricity": 1, "hydrogen": -1 / 0.6},
        hasCapacityVariable=True,
        investPerCapacity=capexRSOC / 2,
        opexPerCapacity=capexRSOC * 0.02 / 2,
        interestRate=0.08,
        economicLifetime=10,
    )
)

# 5. Add commodity storages to the energy system model

## 5.1. Electricity storage

### Lithium ion batteries

The self discharge of a lithium ion battery is here described as 3% per month. The self discharge per hours is obtained using the equation (1-$\text{selfDischarge}_\text{hour})^{30*24\text{h}} = 1-\text{selfDischarge}_\text{month}$.

In [18]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=22,
    )
)

## 5.2. Hydrogen storage

### Hydrogen filled salt caverns
The maximum capacity is here obtained by: dividing the given capacity (which is given for methane) by the lower heating value of methane and then multiplying it with the lower heating value of hydrogen.

In [19]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Salt caverns (hydrogen)",
        commodity="hydrogen",
        hasCapacityVariable=True,
        capacityVariableDomain="continuous",
        capacityPerPlantUnit=133,
        chargeRate=1 / 470.37,
        dischargeRate=1 / 470.37,
        sharedPotentialID="Existing salt caverns",
        stateOfChargeMin=0.33,
        stateOfChargeMax=1,
        capacityMax=data["Salt caverns (hydrogen), capacityMax"],
        investPerCapacity=0.00011,
        opexPerCapacity=0.00057,
        interestRate=0.08,
        economicLifetime=30,
    )
)

## 5.3. Methane storage

### Methane filled salt caverns

In [20]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Salt caverns (biogas)",
        commodity="biogas",
        hasCapacityVariable=True,
        capacityVariableDomain="continuous",
        capacityPerPlantUnit=443,
        chargeRate=1 / 470.37,
        dischargeRate=1 / 470.37,
        sharedPotentialID="Existing salt caverns",
        stateOfChargeMin=0.33,
        stateOfChargeMax=1,
        capacityMax=data["Salt caverns (methane), capacityMax"],
        investPerCapacity=0.00004,
        opexPerCapacity=0.00001,
        interestRate=0.08,
        economicLifetime=30,
    )
)

## 5.4 Pumped hydro storage

### Pumped hydro storage

In [21]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Pumped hydro storage",
        commodity="electricity",
        chargeEfficiency=0.88,
        dischargeEfficiency=0.88,
        hasCapacityVariable=True,
        selfDischarge=1 - (1 - 0.00375) ** (1 / (30 * 24)),
        chargeRate=0.16,
        dischargeRate=0.12,
        capacityFix=data["Pumped hydro storage, capacityFix"],
        investPerCapacity=0,
        opexPerCapacity=0.000153,
    )
)

# 6. Add commodity transmission components to the energy system model

## 6.1. Electricity transmission

### AC cables

esM.add(fn.LinearOptimalPowerFlow(esM=esM, name='AC cables', commodity='electricity',
                                  hasCapacityVariable=True, capacityFix=data['AC cables, capacityFix'],
                                  reactances=data['AC cables, reactances']))

In [22]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="AC cables",
        commodity="electricity",
        hasCapacityVariable=True,
        capacityFix=data["AC cables, capacityFix"],
    )
)

### DC cables

In [23]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="DC cables",
        commodity="electricity",
        losses=data["DC cables, losses"],
        distances=data["DC cables, distances"],
        hasCapacityVariable=True,
        capacityFix=data["DC cables, capacityFix"],
    )
)

## 6.2 Methane transmission

### Methane pipeline

In [24]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="Pipelines (biogas)",
        commodity="biogas",
        distances=data["Pipelines, distances"],
        hasCapacityVariable=True,
        hasIsBuiltBinaryVariable=True,
        bigM=300,
        locationalEligibility=data["Pipelines, eligibility"],
        capacityMax=data["Pipelines, eligibility"] * 15,
        sharedPotentialID="pipelines",
        investPerCapacity=0.000037,
        investIfBuilt=0.000314,
        interestRate=0.08,
        economicLifetime=40,
    )
)

## 6.3 Hydrogen transmission

### Hydrogen pipelines

In [25]:
esM.add(
    fn.Transmission(
        esM=esM,
        name="Pipelines (hydrogen)",
        commodity="hydrogen",
        distances=data["Pipelines, distances"],
        hasCapacityVariable=True,
        hasIsBuiltBinaryVariable=True,
        bigM=300,
        locationalEligibility=data["Pipelines, eligibility"],
        capacityMax=data["Pipelines, eligibility"] * 15,
        sharedPotentialID="pipelines",
        investPerCapacity=0.000177,
        investIfBuilt=0.00033,
        interestRate=0.08,
        economicLifetime=40,
    )
)

# 7. Add commodity sinks to the energy system model

## 7.1. Electricity sinks

### Electricity demand

In [26]:
esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=data["Electricity demand, operationRateFix"],
    )
)

## 7.2. Hydrogen sinks

### Fuel cell electric vehicle (FCEV) demand

In [27]:
FCEV_penetration = 0.5
esM.add(
    fn.Sink(
        esM=esM,
        name="Hydrogen demand",
        commodity="hydrogen",
        hasCapacityVariable=False,
        operationRateFix=data["Hydrogen demand, operationRateFix"] * FCEV_penetration,
    )
)

## 7.3. CO2 sinks

### CO2 exiting the system's boundary

In [28]:
esM.add(
    fn.Sink(
        esM=esM,
        name="CO2 to enviroment",
        commodity="CO2",
        hasCapacityVariable=False,
        balanceLimitID="CO2 limit",
    )
)

All components are now added to the model and the model can be optimized. If the computational complexity of the optimization should be reduced, the time series data of the specified components can be clustered before the optimization and the parameter timeSeriesAggregation is set to True in the optimize call.

# 8. Temporal aggregation

In [ ]:
esM.aggregateTemporally(n_clusters=7)

# 9. Cost-optimal optimization

MGA starts from the cost-optimal solution, so the model is optimized first. The keyword arguments of `esM.optimize` are stored in `solverKwargs` because `fn.runMGA` needs the same ones for every MGA solve. In particular, `timeSeriesAggregation=True` has to be passed there again, otherwise the results of the alternatives are post-processed as if no time series aggregation was used.

In [ ]:
# The `optimizationSpecs` only work with the Gurobi solver. If you are using another solver you need to choose
# specs specific to this solver or no specs.
solverKwargs = dict(
    timeSeriesAggregation=True,
    optimizationSpecs="OptimalityTol=1e-3 method=2 cuts=0 MIPGap=5e-3",
    solver=fn.utils.ImplementedSolvers.GUROBI.value,
)
esM.optimize(**solverKwargs)

# 10. Generate alternatives with MGA

The MGA run is specified by a dictionary:
- `slack`: allowed increase of the total annual cost compared to the cost-optimal solution (here 10 %).
- `objectives`: one entry per alternative. Each objective has a `name` (also used for the Excel file name) and a `sense` (`"min"` or `"max"`). The `variableSelection` defines which variables enter the objective:
    - `"custom"`: only the variables listed in `terms`. A term consists of a `component`, a `variable` (`"capacity"`, `"operation"`, `"charge"` or `"discharge"`) and a `weight`. Optionally, it can be restricted to some `locations` and `investmentPeriods`.
    - `"allCapacity"`: all capacity variables of the model.
    - `"allCapacityAndOperation"`: all capacity and operation variables of the model.

    For `"allCapacity"` and `"allCapacityAndOperation"`, the weights can be adapted with `defaultWeight`, `weights` and `excludeComponents`. Operation variables are summed up to annual values.

Here, two alternatives are generated. Both minimize a weighted sum of the onshore wind capacity, the PV generation, the pumped hydro storage discharge and the transmission capacities. The second alternative puts a higher weight on the storage discharge and the transmission capacities. Note that the operation variables are annual sums (in GWh) and are therefore much larger than the capacities (in GW). With the weights chosen here, the PV generation dominates both objectives. The weights should be chosen with these magnitudes in mind.

With `save_mga_results_in_excel=True`, the cost-optimal solution and every alternative are written to Excel files in the folder `OutputData`. All further keyword arguments (here `solverKwargs`) are passed to `esM.optimize`.

In [ ]:
mgaSpec = {
    "slack": 0.1,
    "objectives": [
        {
            "name": "alternative1",
            "sense": "min",
            "variableSelection": "custom",
            "terms": [
                {"component": "Wind (onshore)", "variable": "capacity", "weight": 0.3},
                {"component": "PV", "variable": "operation", "weight": 0.6},
                {
                    "component": "Pumped hydro storage",
                    "variable": "discharge",
                    "weight": 0.01,
                },
                {"component": "AC cables", "variable": "capacity", "weight": 0.01},
                {"component": "DC cables", "variable": "capacity", "weight": 0.01},
                {
                    "component": "Pipelines (biogas)",
                    "variable": "capacity",
                    "weight": 0.01,
                },
                {
                    "component": "Pipelines (hydrogen)",
                    "variable": "capacity",
                    "weight": 0.01,
                },
            ],
        },
        {
            "name": "alternative2",
            "sense": "min",
            "variableSelection": "custom",
            "terms": [
                {"component": "Wind (onshore)", "variable": "capacity", "weight": 0.3},
                {"component": "PV", "variable": "operation", "weight": 0.6},
                {
                    "component": "Pumped hydro storage",
                    "variable": "discharge",
                    "weight": 0.9,
                },
                {"component": "AC cables", "variable": "capacity", "weight": 0.8},
                {"component": "DC cables", "variable": "capacity", "weight": 0.4},
                {
                    "component": "Pipelines (biogas)",
                    "variable": "capacity",
                    "weight": 0.67,
                },
                {
                    "component": "Pipelines (hydrogen)",
                    "variable": "capacity",
                    "weight": 0.12,
                },
            ],
        },
    ],
}

results = fn.runMGA(
    esM,
    spec=mgaSpec,
    save_mga_results_in_excel=True,
    excelOutputDir="OutputData",
    excelFilePrefix="MGA",
    **solverKwargs,
)

# 11. Compare the alternatives

`fn.runMGA` returns a dictionary with one entry for the cost-optimal solution (`"costOptimal"`) and one entry per alternative. Each entry contains the total annual cost (`"cost"`), the optimization summaries of all modeling classes (`"summary"`) and, for the alternatives, the value of the MGA objective (`"mgaObjective"`).

All alternatives have a total annual cost of at most 110 % of the cost-optimal solution:

In [ ]:
pd.DataFrame(
    {
        name: {
            "TAC [1e9 Euro/a]": result["cost"],
            "MGA objective": result.get("mgaObjective"),
        }
        for name, result in results.items()
    }
)

The installed capacities, summed over all locations, show how differently the alternatives are designed:

In [ ]:
def totalCapacities(summaries):
    """Return the capacity of each component, summed over all locations."""
    capacities = []
    for modelingClass, summary in summaries.items():
        capacity = (
            summary.xs("capacity", level="Property")
            .groupby(level=["Component", "Unit"])
            .sum()
            .sum(axis=1)
        )
        if modelingClass == "TransmissionModel":
            capacity /= 2  # the summary contains both directions of each connection
        capacities.append(capacity)
    return pd.concat(capacities)


capacities = pd.DataFrame(
    {name: totalCapacities(result["summary"]) for name, result in results.items()}
)
capacities.round(2)